# 統計3級-06. 共分散・変動係数・二項分布の正規近似

3級で抜けやすい計算系トピックをまとめて押さえます。
- 共分散（相関係数の前段）
- 変動係数（ばらつきの比較）
- 二項分布の正規近似

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')

## 1. 共分散

2つの量が「一緒に大きくなる／小さくなる」傾向を表す数。
$$ s_{xy} = \frac{1}{n}\sum (x_i - \bar{x})(y_i - \bar{y}) $$

正なら正の関係、負なら負の関係。ただし**単位に依存する**のが弱点。

In [ ]:
x = df['数学'].values
y = df['英語'].values
cov = np.mean((x - x.mean()) * (y - y.mean()))   # 手計算(÷n)
print('共分散:', round(cov, 2))
print('np.cov(÷n-1):', round(np.cov(x, y, ddof=0)[0,1], 2))

## 2. 共分散 → 相関係数

共分散を各標準偏差で割ると、単位によらない −1〜1 の**相関係数**になります。
$$ r = \frac{s_{xy}}{s_x \, s_y} $$

In [ ]:
r = cov / (x.std() * y.std())
print('相関係数(共分散から):', round(r, 3))
print('pandas .corr() で確認:', round(df['数学'].corr(df['英語']), 3))

## 3. 変動係数（CV）

標準偏差を平均で割ったもの。**単位やスケールが違うデータのばらつきを比べる**のに使う。
$$ CV = \frac{標準偏差}{平均} $$

例：身長(cm)と体重(kg)、どちらが相対的にばらつくか。

In [ ]:
height = np.array([158, 162, 170, 165, 168, 160])
weight = np.array([48, 55, 70, 60, 65, 50])
for name, d in [('身長', height), ('体重', weight)]:
    cv = d.std() / d.mean()
    print(f'{name}: 標準偏差{d.std():.1f}, 平均{d.mean():.1f}, CV={cv:.3f}')
print('→ CVが大きい方が、平均に対して相対的にばらついている')

## 4. 二項分布の正規近似

二項分布 $B(n, p)$ は、$n$ が大きいと正規分布 $N(np,\ np(1-p))$ に近づきます。
コインを多数回投げたときの表の回数で確かめます。

In [ ]:
n, p = 100, 0.5
k = np.arange(0, n + 1)
binom = stats.binom.pmf(k, n, p)
mu, sigma = n * p, np.sqrt(n * p * (1 - p))
xs = np.linspace(0, n, 300)
normal = stats.norm.pdf(xs, mu, sigma)

plt.bar(k, binom, alpha=0.5, label='二項分布 B(100,0.5)')
plt.plot(xs, normal, 'r-', label=f'正規近似 N({mu:.0f},{sigma**2:.0f})')
plt.legend(); plt.title('二項分布の正規近似'); plt.show()
print(f'平均 np={mu}, 標準偏差 √(np(1-p))={sigma:.2f}')

### 近似を使った確率計算
「表が60回以上」の確率を、二項分布そのものと正規近似で比べます。

In [ ]:
exact = 1 - stats.binom.cdf(59, n, p)
# 連続補正（59.5を境にする）を入れた正規近似
approx = 1 - stats.norm.cdf(59.5, mu, sigma)
print(f'二項分布での厳密値: {exact:.4f}')
print(f'正規近似(連続補正): {approx:.4f}  → ほぼ一致')

---
## 🏆 練習問題

**問1.** `df` の `勉強時間` と `数学` の共分散を手計算の式で求め、
それを標準偏差で割って相関係数になることを確かめよう。

**問2.** A店の日販 `[50,52,48,51,49]`（万円）、B店 `[10,14,8,12,16]`（万円）。
変動係数を比べ、どちらが相対的に安定しているか答えよう。

**問3.** サイコロを180回ふるとき「1の目」が出る回数は二項分布 B(180, 1/6)。
平均と標準偏差を求め、正規近似で「35回以上出る確率」を計算しよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


<details><summary>解答例</summary>

```python
a=df['勉強時間'].values; b=df['数学'].values
cov=np.mean((a-a.mean())*(b-b.mean())); print(cov/(a.std()*b.std()))

for d in [np.array([50,52,48,51,49]), np.array([10,14,8,12,16])]:
    print(d.std()/d.mean())   # B店の方がCV大＝不安定

mu, sig = 180/6, np.sqrt(180*(1/6)*(5/6))
print(mu, sig, 1-stats.norm.cdf(34.5, mu, sig))
```
</details>

🎉 これで**3級の出題範囲をほぼ全てカバー**しました。